In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

def cramer_2x2(A, b):
    detA = np.linalg.det(A)

    A1 = A.copy()
    A1[:, 0] = b

    A2 = A.copy()
    A2[:, 1] = b

    return np.array([
        np.linalg.det(A1) / detA,
        np.linalg.det(A2) / detA
    ])


def cramer_3x3(A, b):
    detA = np.linalg.det(A)
    x = []

    for i in range(3):
        Ai = A.copy()
        Ai[:, i] = b
        x.append(np.linalg.det(Ai) / detA)

    return np.array(x)


def lu_decomposition(A):
    A = A.astype(float)
    n = A.shape[0]

    L = np.eye(n)
    U = A.copy()
    P = np.eye(n)

    for i in range(n):

        max_row = i + np.argmax(abs(U[i:, i]))
        U[[i, max_row]] = U[[max_row, i]]
        P[[i, max_row]] = P[[max_row, i]]

        for j in range(i+1, n):
            fator = U[j, i] / U[i, i]
            L[j, i] = fator
            U[j, i:] -= fator * U[i, i:]

    return P, L, U


def lu_solve(A, b):
    P, L, U = lu_decomposition(A)

    b = P @ b

    y = np.zeros(len(b))
    for i in range(len(b)):
        y[i] = b[i] - np.sum(L[i, :i] * y[:i])

    x = np.zeros(len(b))
    for i in range(len(b)-1, -1, -1):
        x[i] = (y[i] - np.sum(U[i, i+1:] * x[i+1:])) / U[i, i]

    return x


def benchmark(n):
    A = np.random.rand(n, n)
    b = np.random.rand(n)

    start = time.time()
    np.linalg.solve(A, b)
    t_gauss = time.time() - start

    start = time.time()
    lu_solve(A, b)
    t_lu = time.time() - start

    return t_gauss, t_lu


sizes = [2, 3, 5, 10, 20, 50]

gauss_times = []
lu_times = []

for n in sizes:
    g, l = benchmark(n)
    gauss_times.append(g)
    lu_times.append(l)

plt.plot(sizes, gauss_times, label="Gauss (numpy solve)")
plt.plot(sizes, lu_times, label="LU manual")
plt.xlabel("n")
plt.ylabel("tempo (s)")
plt.title("Benchmark Gauss vs LU")
plt.grid()
plt.legend()
plt.show()


def multi_rhs(A, b_list):
    P, L, U = lu_decomposition(A)

    results = []

    for b in b_list:
        b = P @ b

        y = np.zeros(len(b))
        for i in range(len(b)):
            y[i] = b[i] - np.sum(L[i, :i] * y[:i])

        x = np.zeros(len(b))
        for i in range(len(b)-1, -1, -1):
            x[i] = (y[i] - np.sum(U[i, i+1:] * x[i+1:])) / U[i, i]

        results.append(x)

    return results


def heatmap(A):
    P, L, U = lu_decomposition(A)

    fig, ax = plt.subplots(1, 3, figsize=(12,4))

    ax[0].imshow(A)
    ax[0].set_title("A")

    ax[1].imshow(L)
    ax[1].set_title("L")

    ax[2].imshow(U)
    ax[2].set_title("U")

    plt.tight_layout()
    plt.show()


A = np.array([[3, 2, 1],
              [1, 3, 2],
              [2, 1, 3]], float)

b = np.array([1, 2, 3], float)

print("Cramer 3x3:", cramer_3x3(A, b))
print("LU:", lu_solve(A, b))

b_list = [np.array([1,2,3]), np.array([4,5,6])]
print("Multi RHS:", multi_rhs(A, b_list))

heatmap(A)

NameError: name 'benchmark' is not defined